# Jeopardy Clue Dataset — Exploratory Analysis

Initial exploration of 530k Jeopardy clues (Seasons 1–41, 1984–2025) to assess feasibility of extracting a cultural literacy concept list.

**Key questions:**
1. What does the category landscape look like? How many unique categories, and how do they cluster?
2. How is temporal coverage distributed? Are there gaps or era biases?
3. What do the correct responses ("questions") look like — are they entity-dense enough for concept extraction?
4. Can we identify the most frequently referenced concepts from raw frequency alone?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 1. Load and Inspect

In [ ]:
df = pd.read_csv(
    "../data/jeopardy_clue_dataset/combined_season1-41.tsv",
    sep="\t",
    dtype={"clue_value": "Int64", "daily_double_value": "Int64"},
    parse_dates=["air_date"],
)

print(f"Shape: {df.shape}")
print(f"Date range: {df['air_date'].min()} to {df['air_date'].max()}")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

In [ ]:
df.info()
print("\n--- Null counts ---")
print(df.isnull().sum())
print("\n--- Round distribution ---")
print(df["round"].value_counts())

## 2. Temporal Coverage

How many clues per year/season? Are there gaps?

In [ ]:
df["year"] = df["air_date"].dt.year

clues_per_year = df.groupby("year").size()

fig, ax = plt.subplots()
clues_per_year.plot(kind="bar", ax=ax, color="steelblue", edgecolor="none")
ax.set_title("Jeopardy Clues per Year")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Clues")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

## 3. Category Landscape

Categories are freeform and often punny. How many unique categories exist, and what are the most common?

In [ ]:
n_unique_cats = df["category"].nunique()
print(f"Unique categories: {n_unique_cats:,}")

top_cats = df["category"].value_counts().head(40)
print(f"\nTop 40 categories (by clue count):")
print(top_cats.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
top_cats.plot(kind="barh", ax=ax, color="steelblue", edgecolor="none")
ax.set_title("Top 40 Most Frequent Jeopardy Categories")
ax.set_xlabel("Number of Clues")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Response Analysis

The `question` column contains the correct response — these are the raw concepts. Let's look at what they contain.

In [ ]:
# Basic stats on response text
df["response_len"] = df["question"].str.len()
df["response_word_count"] = df["question"].str.split().str.len()

print("Response length (characters):")
print(df["response_len"].describe())
print("\nResponse word count:")
print(df["response_word_count"].describe())

In [ ]:
# Most frequent correct responses — raw concept frequency
top_responses = df["question"].str.lower().str.strip().value_counts().head(50)
print("Top 50 most frequent correct responses:")
print(top_responses.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
top_responses.head(30).plot(kind="barh", ax=ax, color="steelblue", edgecolor="none")
ax.set_title("Top 30 Most Frequent Correct Responses")
ax.set_xlabel("Frequency")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Concept Seasonality

For the top responses, how consistently do they appear across seasons? Concepts that show up in 30+ of 41 seasons are more durable than ones that cluster in a few years.

In [ ]:
# Derive season from notes or approximate from air_date
# Jeopardy seasons start in September, so season ~ year of September start
df["season"] = df["air_date"].apply(
    lambda d: d.year if d.month >= 9 else d.year - 1
)

# For top 30 responses: in how many distinct seasons do they appear?
df["response_lower"] = df["question"].str.lower().str.strip()
top_30 = top_responses.head(30).index.tolist()

season_spread = (
    df[df["response_lower"].isin(top_30)]
    .groupby("response_lower")["season"]
    .nunique()
    .reindex(top_30)
)

spread_df = pd.DataFrame({
    "total_count": top_responses.head(30),
    "seasons_appeared": season_spread,
})
spread_df["avg_per_season"] = spread_df["total_count"] / spread_df["seasons_appeared"]

print("Top 30 responses — frequency and seasonal spread:")
print(spread_df.to_string())

## 6. Category Domain Clustering (Preview)

Many categories are puns or variations of the same domain (e.g., 'AMERICAN HISTORY', 'U.S. HISTORY', 'AMERICANA'). A quick look at how categories might cluster using simple keyword matching.

In [ ]:
# Simple keyword-based domain tagging
domain_keywords = {
    "History": ["history", "historic", "civil war", "world war", "revolution", "ancient", "medieval"],
    "Science": ["science", "biology", "chemistry", "physics", "astronomy", "medicine", "anatomy"],
    "Literature": ["literature", "authors", "novels", "poetry", "poets", "books", "shakespeare"],
    "Geography": ["geography", "countries", "capitals", "rivers", "lakes", "islands", "mountains", "continents"],
    "Arts": ["art", "painting", "music", "composers", "opera", "ballet", "sculpture", "classical"],
    "Film & TV": ["movies", "film", "television", "tv ", "actors", "actresses", "oscar", "emmy"],
    "Politics & Government": ["president", "politics", "government", "congress", "supreme court", "constitution"],
    "Religion & Mythology": ["bible", "religion", "mythology", "gods", "church"],
    "Language & Words": ["word", "language", "vocabulary", "letter", "spelling", "grammar"],
    "Sports": ["sports", "baseball", "football", "basketball", "olympics", "hockey", "tennis", "golf"],
}

cat_lower = df["category"].str.lower()

domain_counts = {}
for domain, keywords in domain_keywords.items():
    mask = cat_lower.str.contains("|".join(keywords), na=False)
    domain_counts[domain] = mask.sum()

domain_counts["Other/Unclassified"] = len(df) - sum(domain_counts.values())

domain_series = pd.Series(domain_counts).sort_values(ascending=False)
print("Approximate domain distribution:")
print(domain_series.to_string())
print(f"\nClassified: {sum(domain_counts.values()) - domain_counts['Other/Unclassified']:,} / {len(df):,} ({100*(sum(domain_counts.values()) - domain_counts['Other/Unclassified'])/len(df):.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
domain_series.plot(kind="barh", ax=ax, color="steelblue", edgecolor="none")
ax.set_title("Jeopardy Clues by Domain (Keyword Classification)")
ax.set_xlabel("Number of Clues")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Next Steps

1. **NER-based concept extraction** — Run spaCy NER on clue text and responses to extract entities, then link to Wikidata
2. **Category normalization** — Cluster the ~30k+ unique categories into a manageable taxonomy
3. **Concept importance scoring** — Combine frequency, seasonal spread, category breadth, and dollar value into a single importance metric
4. **Gap analysis** — Compare extracted concept list against Hirsch's original to identify coverage and gaps
5. **Supplementary data sources** — Identify APIs/datasets for curriculum standards, Wikipedia importance, etc.